In [ ]:
# =================================================================
# Phase 4: 资产定价终极测试 (Asset Pricing Tests)
# 1. 10分位组合排序 (Decile Sorts)
# 2. Fama-MacBeth 截面回归 (Cross-Sectional Regression)
# 3. 投资组合 Alpha 测试 (Time-Series Factor Spanning Test)
# =================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from scipy.stats.mstats import winsorize
import warnings
import sqlite3
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
warnings.filterwarnings('ignore')

print("=== Starting Phase 4: Institutional Asset Pricing Tests ===")
db_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
# =================================================================
# 第一步：数据对齐与严格清洗 (Data Merging & Cleaning)
# =================================================================
print("\n>>> 1. Merging Data and Applying Universe Filters...")

micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/DCDE/data/micro_data.parquet')
df_factors = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/DCDE/data/final_pricing_factors.parquet')

with sqlite3.connect(db_path) as conn:
    # read ff_factors_monthly and ff_fivefactors_monthly
    ff_factors_monthly = pd.read_sql_query("SELECT * FROM ff_factors_monthly", conn)
    ff_fivefactors_monthly = pd.read_sql_query("SELECT * FROM ff_fivefactors_monthly", conn)

ff_factor_method = 5

if ff_factor_method == 3:
    # fama french 3 factors + momentum
    ff_df = ff_factors_monthly
    ff_factors = ['mktrf','smb','hml','umd']
elif ff_factor_method == 5:
    # fama french 5 factors + momentum
    ff_df = ff_fivefactors_monthly
    ff_factors = ['mktrf','smb','hml','umd','rmw','cma']


ff_df['date'] = pd.to_datetime(ff_df['date'])+pd.offsets.MonthEnd(0)


# 1. 对齐日期列进行合并
# df_factors 的日期叫 'date', micro_df 的日期叫 'mthcaldt'
df_micro_subset = micro_df[['permno', 'mthcaldt', 'mthprc', 'mthcap', 'mthcap_log', 'bm', 'MOM12m', 'mthret_lead1']]
df = pd.merge(df_factors, df_micro_subset, 
              left_on=['permno', 'date'], 
              right_on=['permno', 'mthcaldt'], 
              how='inner')

# 2. 清理缺失值：确保用于排序、加权和回归的核心列没有 NaN
cols_to_dropna = ['es_5', 'mthret_lead1', 'mthcap', 'mthprc', 'mthcap_log', 'bm', 'MOM12m']
df = df.dropna(subset=cols_to_dropna).copy()

# 3. 仙股过滤 (Penny Stock Filter)：剔除建仓当月价格低于 5 美元的股票
# CRSP 数据库中负价格代表买卖中间价，必须取绝对值
df['price_abs'] = np.abs(df['mthprc'])
initial_len = len(df)
df = df[df['price_abs'] >= 5.0]
print(f"    -> Filtered {initial_len - len(df)} penny stock observations (< $5).")

# =================================================================
# 第二步：市值加权 10 分位测试 (Value-Weighted Decile Sorts)
# =================================================================
print("\n>>> 2. Running Value-Weighted Portfolio Sorts...")

# 1. 按月对预期尾部风险 (es_5) 进行 10 等分
def get_deciles(x):
    # return pd.qcut(x, 10, labels=False, duplicates='drop') + 1
    # 在 pd.qcut 的默认设置下，第 10 组的 es_5 最大，第 1 组的 es_5 最小。
    return pd.qcut(x, 10, labels=False) + 1

df['decile'] = df.groupby('date')['es_5'].transform(get_deciles)

# 2. 计算市值加权收益率 (VW Returns)
def vw_return(group):
    if group['mthcap'].sum() == 0: 
        return np.nan
    # 使用 t 时刻的市值 (mthcap) 作为权重，加权 t+1 期的真实收益率 (mthret_lead1)
    return np.average(group['mthret_lead1'], weights=group['mthcap'])

# 计算各组每月的加权收益率
vw_ret = df.groupby(['date', 'decile']).apply(vw_return).unstack()

# 构建多空组合 (Long-Short Portfolio)：做多风险最高组(10)，做空风险最低组(1)
# 这里 10 代表尾部风险最大
vw_ret['High-Low'] = vw_ret[10] - vw_ret[1]

# 3. 统计输出表
print("\n--- Value-Weighted Decile Portfolios (Monthly Returns %) ---")
summary_vw = pd.DataFrame(index=vw_ret.columns)
summary_vw['Mean (Ann. %)'] = vw_ret.mean() * 12 * 100
summary_vw['Vol (Ann. %)']  = vw_ret.std() * np.sqrt(12) * 100
summary_vw['Sharpe']        = summary_vw['Mean (Ann. %)'] / summary_vw['Vol (Ann. %)']
# 计算均值是否显著异于 0 的 t 统计量
def get_nw_tstat(x, lags=None):
    # 自动选择滞后阶数，常用的是 lags=int(4*(n/100)**(2/9))
    x = x.dropna()
    n = len(x)
    if lags is None: lags = int(n**(1/4)) 
    
    # 对常数项回归，获取均值及其 Newey-West 标准误
    model = sm.OLS(x, np.ones(n)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return model.tvalues[0]

# 应用到你的数据上
summary_vw['t-stat'] = vw_ret.apply(get_nw_tstat)

print(summary_vw.round(2))


# =================================================================
# 第三步：Fama-MacBeth 截面回归 (Fama-MacBeth Regressions)
# =================================================================
print("\n>>> 3. Running Robust Fama-MacBeth Regressions...")

# def run_cs_reg(df_t):
#     # 截面记录太少则跳过
#     if len(df_t) < 50: return None
    
#     # 🌟 极值缩尾：将 t+1 期真实收益率在上下 1% 分位数进行截断，拯救 OLS
#     df_t['ret_lead1_win'] = winsorize(df_t['mthret_lead1'], limits=[0.01, 0.01])
    
#     # 提取自变量并进行截面标准化 (Z-score)，使得不同量纲的系数具有可比性
#     X = df_t[['es_5', 'mthcap_log', 'bm', 'MOM12m']]
#     X = (X - X.mean()) / X.std()
#     X = sm.add_constant(X)
    
#     # 因变量转为 % 方便阅读
#     y = df_t['ret_lead1_win'] * 100
    
#     model = sm.OLS(y, X).fit()
#     return model.params

# # 逐月执行截面回归
# fm_coefs = df.groupby('date').apply(run_cs_reg).dropna()

# # 时序汇总：使用 Newey-West 调整 (Lag=6) 消除时间序列自相关带来的标准误低估
# results = []
# for col in fm_coefs.columns:
#     res = sm.OLS(fm_coefs[col], np.ones(len(fm_coefs))).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
#     results.append({
#         'Factor': col,
#         'Coefficient': res.params[0],
#         't-stat': res.tvalues[0],
#         'p-value': res.pvalues[0]
#     })

# fm_summary = pd.DataFrame(results).set_index('Factor')

# Step 1: Cross-sectional regressions
def run_cs_reg(df_t):
    if len(df_t) < 50:
        return None
    
    df_t = df_t.copy()
    
    # winsorize (safer than scipy)
    lower = df_t['mthret_lead1'].quantile(0.01)
    upper = df_t['mthret_lead1'].quantile(0.99)
    y = df_t['mthret_lead1'].clip(lower, upper) * 100
    
    X = df_t[['es_5', 'mthcap_log', 'bm', 'MOM12m']]
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    return model.params

fm_coefs = df.groupby('date').apply(run_cs_reg).dropna()

# Step 2: Newey-West adjusted inference
T = len(fm_coefs)
lag = int(4 * (T / 100) ** (2/9))   # 更规范的lag选择

results = []
for col in fm_coefs.columns:
    y = fm_coefs[col]
    
    res = sm.OLS(y, np.ones(len(y))).fit(
        cov_type='HAC',
        cov_kwds={'maxlags': lag}
    )
    
    results.append({
        'Factor': col,
        'Coefficient': res.params[0],
        't-stat': res.tvalues[0],
        'p-value': res.pvalues[0]
    })

fm_summary = pd.DataFrame(results).set_index('Factor')


print("\n--- Fama-MacBeth Regression Results ---")
print("Dep. Variable: Next Month Return (%) [Winsorized]")
print("Newey-West HAC Adjusted t-statistics (Lag=6)")
print("-" * 50)
print(fm_summary.round(3))
print("-" * 50)


# =================================================================
# 第四步：投资组合 Alpha 测试 (Time-Series Factor Spanning Test)
# =================================================================
print("\n>>> 4. Running Portfolio Alpha Test (Fama-French 5-Factor)...")

# 提取我们的 High-Low 零成本多空投资组合的时间序列收益率
port_ret = vw_ret[['High-Low']].dropna().reset_index()
port_ret.columns = ['date', 'High_Low_Ret']

# 对齐日期并合并 (必须使用 inner join 确保时间严丝合缝)
alpha_df = pd.merge(port_ret, ff_df, on='date', how='inner')

# 构造回归模型：
# 因为 High-Low 是做多做空的“零投资组合”，所以左侧因变量不需要再减去无风险利率 (RF)
# y = High_Low_Ret
# X = 'mktrf','smb','hml','umd', 'rmw','cma'

y_alpha = alpha_df['High_Low_Ret']

X_alpha = alpha_df[ff_factors]
X_alpha = sm.add_constant(X_alpha) # 截距项就是我们要找的 Alpha！

# 进行时序回归 (同样使用 Newey-West HAC 调整)
alpha_model = sm.OLS(y_alpha, X_alpha).fit(cov_type='HAC', cov_kwds={'maxlags': 6})

print("\n--- Fama-French 5-Factor Time-Series Regression ---")
print(alpha_model.summary().tables[1]) # 只打印核心的系数表

alpha_val = alpha_model.params['const']
alpha_t = alpha_model.tvalues['const']
print(f"\n🌟 Monthly FF5 Alpha: {alpha_val:.3f}% (t-stat: {alpha_t:.2f})")
print(f"🌟 Annualized Alpha : {alpha_val * 12:.2f}%")


print("\n=== Phase 4 Complete! ===")

=== Starting Phase 4: Institutional Asset Pricing Tests ===

>>> 1. Merging Data and Applying Universe Filters...
    -> Filtered 145305 penny stock observations (< $5).

>>> 2. Running Value-Weighted Portfolio Sorts...

--- Value-Weighted Decile Portfolios (Monthly Returns %) ---
          Mean (Ann. %)  Vol (Ann. %)  Sharpe  t-stat
decile                                               
1                 12.97         13.37    0.97    4.68
2                 11.79         16.04    0.73    3.26
3                 12.52         17.56    0.71    3.04
4                  8.40         19.41    0.43    1.81
5                  8.20         20.95    0.39    1.65
6                  9.54         22.37    0.43    1.73
7                  9.11         24.22    0.38    1.55
8                  5.42         27.51    0.20    0.77
9                  8.92         31.61    0.28    1.11
10                13.31         37.51    0.35    1.34
High-Low           0.34         31.40    0.01    0.04

>>> 3. Running 